In [2]:
import pandas as pd

lien = "https://datasets.imdbws.com/title.akas.tsv.gz"
akas = pd.read_csv(lien, sep="\t", na_values="\\N")

C:\Users\bourr\AppData\Local\Temp\ipykernel_17292\3375504777.py:4: DtypeWarning: Columns (6: attributes) have mixed types. Specify dtype option on import or set low_memory=False.
  akas = pd.read_csv(lien, sep="\t", na_values="\\N")


In [3]:
akas.head() # pour vérification des titres de colonnes et des données

,titleId,ordering,title,region,language,types,attributes,isOriginalTitle
0,tt0000001,1,Carmencita,NaN,NaN,original,NaN,1
1,tt0000001,2,Carmencita,DE,NaN,NaN,literal title,0
2,tt0000001,3,Carmencita,US,NaN,imdbDisplay,NaN,0
3,tt0000001,4,Carmencita - spanyol tánc,HU,NaN,imdbDisplay,NaN,0
4,tt0000001,5,Καρμενσίτα,GR,NaN,imdbDisplay,NaN,0


In [4]:
# récupération des données filtrées
films = pd.read_csv("../data/films_filtres.csv")
films.head()

,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes
0,tt0035423,movie,Kate & Leopold,0,2001.0,118.0,"Comedy,Fantasy,Romance",6.4,95169
1,tt0064106,movie,The Butcher,0,1970.0,93.0,"Drama,Mystery,Thriller",7.3,10675
2,tt0064177,movie,Colossus: The Forbin Project,0,1970.0,100.0,"Sci-Fi,Thriller",7.1,12058
3,tt0065134,movie,Two Mules for Sister Sara,0,1970.0,116.0,"Adventure,Drama,Romance",7.0,33505
4,tt0065143,movie,The Bird with the Crystal Plumage,0,1970.0,96.0,"Drama,Horror,Mystery",7.1,27767


In [5]:
# tri des données sur titres en français et titres présents dans la liste filtrée

colonnes = ['titleId', 'title', 'region']

nos_films = set(films['tconst'])

akas = pd.read_csv(
    lien,
    sep="\t",
    na_values="\\N",
    usecols=colonnes,
    chunksize=2_000_000,
    )

titres_gardes = []
for morceau in akas:
    est_fr = morceau["region"] == "FR"
    est_a_nous = morceau["titleId"].isin(nos_films)
    titres_gardes.append(morceau[est_fr & est_a_nous])

akas_fr = pd.concat(titres_gardes, ignore_index=True).drop_duplicates()

In [6]:
akas_fr.head() # pour vérification des titres de colonnes et des données

,titleId,title,region
0,tt0035423,Kate et Léopold,FR
1,tt0064106,Le boucher,FR
2,tt0064177,Le cerveau d'acier,FR
3,tt0065134,Sierra torride,FR
4,tt0065143,L'oiseau au plumage de cristal,FR


In [7]:
akas_fr.shape # pour vérifier le nombre de films dans le fichier ==> il y a plus de lignes que dans la liste filtrée

(6589, 3)

In [8]:
akas_fr.duplicated("titleId").sum()

np.int64(608)

In [9]:
# supression des Id-titres en doublons
akas_fr = akas_fr.drop_duplicates(subset="titleId")

In [10]:
akas_fr.shape

(5981, 3)

In [11]:
akas_fr[akas_fr.duplicated("title")] # pour voir le nombre de films ayant les mêmes titres

,titleId,title,region
807,tt0091223,House,FR
1114,tt0099012,Alice,FR
1293,tt0103874,Dracula,FR
1446,tt0108333,Les trois mousquetaires,FR
2055,tt0149261,Peur bleue,FR
...,...,...,...
6204,tt6793580,Champions,FR
6300,tt7390588,Little Monsters,FR
6399,tt8097030,Alerte rouge,FR
6493,tt8946378,À couteaux tirés,FR


In [12]:
# liaison entre les données akas_fr et nos_films
films_francais = pd.merge(
    left=films,
    right=akas_fr,
    how="left",
    left_on="tconst",
    right_on="titleId"
).drop(columns=["titleId", "region"])

films_francais.head()

,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,title
0,tt0035423,movie,Kate & Leopold,0,2001.0,118.0,"Comedy,Fantasy,Romance",6.4,95169,Kate et Léopold
1,tt0064106,movie,The Butcher,0,1970.0,93.0,"Drama,Mystery,Thriller",7.3,10675,Le boucher
2,tt0064177,movie,Colossus: The Forbin Project,0,1970.0,100.0,"Sci-Fi,Thriller",7.1,12058,Le cerveau d'acier
3,tt0065134,movie,Two Mules for Sister Sara,0,1970.0,116.0,"Adventure,Drama,Romance",7.0,33505,Sierra torride
4,tt0065143,movie,The Bird with the Crystal Plumage,0,1970.0,96.0,"Drama,Horror,Mystery",7.1,27767,L'oiseau au plumage de cristal


In [13]:
films_francais.shape

(6447, 10)

In [14]:
films_francais[films_francais["title"].isna()]

,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,title
164,tt0072783,movie,Chupke Chupke,0,1975.0,146.0,"Comedy,Drama,Romance",8.3,14189,NaN
270,tt0077451,movie,Don,0,1978.0,166.0,"Action,Crime,Thriller",7.7,13685,NaN
310,tt0079221,movie,Gol Maal,0,1979.0,120.0,"Comedy,Romance",8.5,22413,NaN
377,tt0081590,movie,The Charter Trip,0,1980.0,107.0,Comedy,7.2,10096,NaN
380,tt0081748,movie,Where the Buffalo Roam,0,1980.0,99.0,"Biography,Comedy",6.5,13409,NaN
...,...,...,...,...,...,...,...,...,...,...
6366,tt9098938,movie,Sharmaji Namkeen,0,2022.0,119.0,"Comedy,Drama,Family",7.4,13915,NaN
6381,tt9248940,movie,Dream Girl,0,2019.0,132.0,"Comedy,Romance",7.0,18997,NaN
6397,tt9420648,movie,Bala,0,2019.0,133.0,Comedy,7.3,18495,NaN
6404,tt9537292,movie,Ala Vaikunthapurramuloo,0,2020.0,163.0,"Action,Comedy,Drama",7.3,27426,NaN


In [15]:
films_francais[films_francais["primaryTitle"].str.contains("fight", case=False)]

,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,title
553,tt0087597,movie,The Last Starfighter,0,1984.0,100.0,"Action,Adventure,Sci-Fi",6.7,50611,Starfighter
1031,tt0101748,movie,Dogfight,0,1991.0,94.0,"Comedy,Drama,Romance",7.3,11514,L'amour n'est pas un jeu
1414,tt0114563,movie,Street Fighter II: The Animated Movie,0,1994.0,102.0,"Action,Adventure,Animation",7.2,11768,Street Fighter II: Le film
1768,tt0137523,movie,Fight Club,0,1999.0,139.0,"Crime,Drama,Thriller",8.8,2683106,Fight Club
2743,tt0436971,movie,Why We Fight,0,2005.0,98.0,"Documentary,History,War",8.0,10557,Le Nerf de la guerre
3188,tt0964517,movie,The Fighter,0,2010.0,116.0,"Action,Biography,Drama",7.8,412630,Fighter
3488,tt11456054,movie,Run Hide Fight,0,2020.0,109.0,"Action,Crime,Drama",6.3,31278,Run Hide Fight
3861,tt13652286,movie,Fight or Flight,0,2024.0,102.0,"Action,Comedy",6.4,29377,Fight or Flight
3881,tt13818368,movie,Fighter,0,2024.0,166.0,"Action,Drama",6.1,54832,Fighter
4936,tt2495118,movie,Ip Man: The Final Fight,0,2013.0,100.0,"Action,Biography,Drama",6.1,11928,Ip Man: Le combat final


In [22]:
# récupération du fichier tmdb
films_tmdb = pd.read_csv("../data/TMDB_movie_dataset_v11.csv", usecols=["imdb_id", "original_language", "poster_path"])

films_tmdb.head()

,imdb_id,original_language,poster_path
0,tt1375666,en,/oYuLEt3zVCKq57qu2F8dT7NIa6f.jpg
1,tt0816692,en,/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg
2,tt0468569,en,/qJ2tW6WMUDux911r6m7haRef0WH.jpg
3,tt0499549,en,/kyeqWdyUXW608qlYkRqosgbbJyK.jpg
4,tt0848228,en,/RYMX2wcKCBAr24UyPD7xwmjaTn.jpg


In [23]:
# fusion avec films_francais
films_francais = pd.merge(
    left=films_francais,
    right=films_tmdb,
    how="left",
    left_on="tconst",
    right_on="imdb_id"
).drop(columns=["imdb_id"])

films_francais.head()

,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,title,original_language,poster_path
0,tt0035423,movie,Kate & Leopold,0,2001.0,118.0,"Comedy,Fantasy,Romance",6.4,95169,Kate et Léopold,en,/mUvikzKJJSg9khrVdxK8kg3TMHA.jpg
1,tt0064106,movie,The Butcher,0,1970.0,93.0,"Drama,Mystery,Thriller",7.3,10675,Le boucher,fr,/An2gkXn5TATag9Adul2Zbq5Ndrv.jpg
2,tt0064177,movie,Colossus: The Forbin Project,0,1970.0,100.0,"Sci-Fi,Thriller",7.1,12058,Le cerveau d'acier,en,/kmVDlTzERxSvBrqtQurjKf1zEy1.jpg
3,tt0065134,movie,Two Mules for Sister Sara,0,1970.0,116.0,"Adventure,Drama,Romance",7.0,33505,Sierra torride,en,/RbuNHMlrw7M2w1R3hZMAPwh6dG.jpg
4,tt0065143,movie,The Bird with the Crystal Plumage,0,1970.0,96.0,"Drama,Horror,Mystery",7.1,27767,L'oiseau au plumage de cristal,it,/dpALRkwodEIT7TGX9SqTgwVNn0i.jpg


In [ ]:
# films qui n'ont ni affiche ni titre français
films_francais[films_francais["poster_path"].isna() & films_francais["title"].isna()]

,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,title,original_language,poster_path
3991,tt14418234,movie,State of Siege: Temple Attack,0,2021.0,110.0,"Action,Drama,Thriller",7.5,10603,NaN,en,NaN
4097,tt15176374,movie,Unpaarvayil,0,2025.0,117.0,Thriller,9.0,10057,NaN,ta,NaN
4164,tt15576290,movie,Do Aur Do Pyaar,0,2024.0,140.0,"Comedy,Drama,Romance",6.4,12432,NaN,NaN,NaN
4696,tt21626774,movie,Sector 36,0,2024.0,123.0,"Crime,Drama,Thriller",7.1,27629,NaN,en,NaN
5084,tt27866781,movie,Kaalidhar Laapata,0,2025.0,109.0,"Comedy,Drama",7.5,10361,NaN,NaN,NaN
5106,tt28089700,movie,Raid 2,0,2025.0,145.0,"Crime,Drama,Thriller",6.6,37238,NaN,NaN,NaN
5139,tt28717622,movie,Bomb,0,2025.0,139.0,"Comedy,Drama",6.1,15027,NaN,NaN,NaN
5236,tt30798612,movie,Crazxy,0,2025.0,93.0,"Drama,Thriller",6.6,18139,NaN,hi,NaN
5243,tt30889755,movie,The Sabarmati Report,0,2024.0,127.0,"Crime,Drama,Thriller",6.4,25877,NaN,NaN,NaN
5261,tt31349844,movie,Runner,0,2026.0,98.0,"Action,Comedy,Thriller",6.2,20348,NaN,NaN,NaN


In [ ]:
# films qui n'ont ni titre français ni original_language
films_francais[films_francais["original_language"].isna() & films_francais["title"].isna()]

,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,title,original_language,poster_path
4164,tt15576290,movie,Do Aur Do Pyaar,0,2024.0,140.0,"Comedy,Drama,Romance",6.4,12432,NaN,NaN,NaN
5084,tt27866781,movie,Kaalidhar Laapata,0,2025.0,109.0,"Comedy,Drama",7.5,10361,NaN,NaN,NaN
5106,tt28089700,movie,Raid 2,0,2025.0,145.0,"Crime,Drama,Thriller",6.6,37238,NaN,NaN,NaN
5139,tt28717622,movie,Bomb,0,2025.0,139.0,"Comedy,Drama",6.1,15027,NaN,NaN,NaN
5243,tt30889755,movie,The Sabarmati Report,0,2024.0,127.0,"Crime,Drama,Thriller",6.4,25877,NaN,NaN,NaN
5261,tt31349844,movie,Runner,0,2026.0,98.0,"Action,Comedy,Thriller",6.2,20348,NaN,NaN,NaN
5433,tt34936199,movie,Daagi,0,2025.0,142.0,"Action,Romance",6.1,11317,NaN,NaN,NaN
5535,tt37809949,movie,Dulhaniya Le Aaeegi,0,2026.0,127.0,Comedy,7.4,10600,NaN,NaN,NaN
5562,tt38525492,movie,Bhagwat Chapter One: Raakshas,0,2025.0,127.0,"Crime,Thriller",6.8,10509,NaN,NaN,NaN
5577,tt39122441,movie,Jockey,0,2026.0,147.0,"Action,Adventure,Drama",7.7,14752,NaN,NaN,NaN


In [ ]:
# films qui ont une affiche, pas de titre en français mais une langue orignale en anglais
films_francais[~films_francais["poster_path"].isna() & films_francais["title"].isna() & films_francais["original_language"].str.contains("en")]

,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,title,original_language,poster_path
380,tt0081748,movie,Where the Buffalo Roam,0,1980.0,99.0,"Biography,Comedy",6.5,13409,NaN,en,/vRMeb4LN416PGvSgG0n063bc5ez.jpg
447,tt0083869,movie,Eating Raoul,0,1982.0,90.0,"Comedy,Crime",6.8,10137,NaN,en,/1B37isseQEN81emH0nSzuMWOCyl.jpg
462,tt0084509,movie,The Plague Dogs,0,1982.0,103.0,"Adventure,Animation,Drama",7.7,10327,NaN,en,/97Z56DEjHNNQXOPW9s9rL15khjM.jpg
513,tt0086373,movie,Strange Brew,0,1983.0,90.0,"Comedy,Crime,Sci-Fi",6.6,20448,NaN,en,/rScplcvRHX7o7XK88SFb5mddo8y.jpg
820,tt0095188,movie,Funny Farm,0,1988.0,101.0,Comedy,6.2,25900,NaN,en,/tW6LGDEZrJpLbvEGkHk51MyQqJC.jpg
...,...,...,...,...,...,...,...,...,...,...,...,...
5707,tt4669186,movie,Kevin Hart: What Now?,0,2016.0,96.0,"Comedy,Documentary",6.0,10364,NaN,en,/lFg4tx0jAYkgrcm1bPBlM0FbQvH.jpg
5720,tt4714568,movie,The Young Offenders,0,2016.0,83.0,"Adventure,Comedy,Crime",7.1,14022,NaN,en,/qpRz7J6Jw4M7Oa02dCRpot7gNO2.jpg
5932,tt5929750,movie,Beatriz at Dinner,0,2017.0,82.0,"Comedy,Drama",6.0,11499,NaN,en,/ypvlGBvm6p0EIvKCzJw0OqsuzK5.jpg
6097,tt6952960,movie,The Kindergarten Teacher,0,2018.0,96.0,"Drama,Thriller",6.7,19075,NaN,en,/pwavkdZb2xNRh5Y9MHhkZJ2UaHT.jpg


In [ ]:
# filtre final à retenir
# films avec un titre français ou avec une affiche et une langue originale en anglais
films_francais[(~films_francais["poster_path"].isna() & films_francais["original_language"].str.contains("en")) | ~films_francais["title"].isna()]

,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,title,original_language,poster_path
0,tt0035423,movie,Kate & Leopold,0,2001.0,118.0,"Comedy,Fantasy,Romance",6.4,95169,Kate et Léopold,en,/mUvikzKJJSg9khrVdxK8kg3TMHA.jpg
1,tt0064106,movie,The Butcher,0,1970.0,93.0,"Drama,Mystery,Thriller",7.3,10675,Le boucher,fr,/An2gkXn5TATag9Adul2Zbq5Ndrv.jpg
2,tt0064177,movie,Colossus: The Forbin Project,0,1970.0,100.0,"Sci-Fi,Thriller",7.1,12058,Le cerveau d'acier,en,/kmVDlTzERxSvBrqtQurjKf1zEy1.jpg
3,tt0065134,movie,Two Mules for Sister Sara,0,1970.0,116.0,"Adventure,Drama,Romance",7.0,33505,Sierra torride,en,/RbuNHMlrw7M2w1R3hZMAPwh6dG.jpg
4,tt0065143,movie,The Bird with the Crystal Plumage,0,1970.0,96.0,"Drama,Horror,Mystery",7.1,27767,L'oiseau au plumage de cristal,it,/dpALRkwodEIT7TGX9SqTgwVNn0i.jpg
...,...,...,...,...,...,...,...,...,...,...,...,...
6445,tt9873892,movie,They Cloned Tyrone,0,2023.0,122.0,"Comedy,Mystery,Sci-Fi",6.7,49479,Ils ont cloné Tyrone,en,/omSa3oafsV7i5776FLFoOBI2Ho9.jpg
6447,tt9893250,movie,I Care a Lot,0,2020.0,118.0,"Comedy,Crime,Drama",6.4,161411,I Care a Lot,en,/gKnhEsjNefpKnUdAkn7INzIFLSu.jpg
6448,tt9894470,movie,VFW,0,2019.0,92.0,"Action,Crime,Horror",6.1,10551,VFW,en,/AnVD7Gn14uOTQhcc5xYZGQ9DRvS.jpg
6449,tt9900782,movie,Kaithi,0,2019.0,145.0,"Action,Crime,Thriller",8.4,53416,Kaithi,ta,/hOF9CgPsy9aLr5GJEBESC8MEXFy.jpg


In [37]:
# création de la sélection
selection = films_francais[~films_francais["title"].isna() | (~films_francais["poster_path"].isna() & films_francais["original_language"].str.contains("en"))]

selection.info()

<class 'pandas.DataFrame'>
Index: 6092 entries, 0 to 6450
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   tconst             6092 non-null   str    
 1   titleType          6092 non-null   str    
 2   primaryTitle       6092 non-null   str    
 3   isAdult            6092 non-null   int64  
 4   startYear          6092 non-null   float64
 5   runtimeMinutes     6092 non-null   float64
 6   genres             6092 non-null   str    
 7   averageRating      6092 non-null   float64
 8   numVotes           6092 non-null   int64  
 9   title              5984 non-null   str    
 10  original_language  5912 non-null   str    
 11  poster_path        5796 non-null   str    
dtypes: float64(3), int64(2), str(7)
memory usage: 618.7 KB


In [39]:
# remplacement des titres français sans valeur par les titres originaux
selection["sans_titre"] = selection["title"].isna()

selection["title"] = selection["title"].fillna(selection["primaryTitle"])

selection[selection["sans_titre"]]

,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,title,original_language,poster_path,sans_titre
380,tt0081748,movie,Where the Buffalo Roam,0,1980.0,99.0,"Biography,Comedy",6.5,13409,Where the Buffalo Roam,en,/vRMeb4LN416PGvSgG0n063bc5ez.jpg,True
447,tt0083869,movie,Eating Raoul,0,1982.0,90.0,"Comedy,Crime",6.8,10137,Eating Raoul,en,/1B37isseQEN81emH0nSzuMWOCyl.jpg,True
462,tt0084509,movie,The Plague Dogs,0,1982.0,103.0,"Adventure,Animation,Drama",7.7,10327,The Plague Dogs,en,/97Z56DEjHNNQXOPW9s9rL15khjM.jpg,True
513,tt0086373,movie,Strange Brew,0,1983.0,90.0,"Comedy,Crime,Sci-Fi",6.6,20448,Strange Brew,en,/rScplcvRHX7o7XK88SFb5mddo8y.jpg,True
820,tt0095188,movie,Funny Farm,0,1988.0,101.0,Comedy,6.2,25900,Funny Farm,en,/tW6LGDEZrJpLbvEGkHk51MyQqJC.jpg,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5707,tt4669186,movie,Kevin Hart: What Now?,0,2016.0,96.0,"Comedy,Documentary",6.0,10364,Kevin Hart: What Now?,en,/lFg4tx0jAYkgrcm1bPBlM0FbQvH.jpg,True
5720,tt4714568,movie,The Young Offenders,0,2016.0,83.0,"Adventure,Comedy,Crime",7.1,14022,The Young Offenders,en,/qpRz7J6Jw4M7Oa02dCRpot7gNO2.jpg,True
5932,tt5929750,movie,Beatriz at Dinner,0,2017.0,82.0,"Comedy,Drama",6.0,11499,Beatriz at Dinner,en,/ypvlGBvm6p0EIvKCzJw0OqsuzK5.jpg,True
6097,tt6952960,movie,The Kindergarten Teacher,0,2018.0,96.0,"Drama,Thriller",6.7,19075,The Kindergarten Teacher,en,/pwavkdZb2xNRh5Y9MHhkZJ2UaHT.jpg,True


In [42]:
selection.info()

<class 'pandas.DataFrame'>
Index: 6092 entries, 0 to 6450
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   tconst             6092 non-null   str    
 1   titleType          6092 non-null   str    
 2   primaryTitle       6092 non-null   str    
 3   isAdult            6092 non-null   int64  
 4   startYear          6092 non-null   float64
 5   runtimeMinutes     6092 non-null   float64
 6   genres             6092 non-null   str    
 7   averageRating      6092 non-null   float64
 8   numVotes           6092 non-null   int64  
 9   title              6092 non-null   str    
 10  original_language  5912 non-null   str    
 11  poster_path        5796 non-null   str    
 12  sans_titre         6092 non-null   bool   
dtypes: bool(1), float64(3), int64(2), str(7)
memory usage: 624.7 KB


In [ ]:
# création du fichier csv des films sélectionnés (reste à affiner)
selection.to_csv("../data/films_selection.csv", index=False)